# M2-B2 — Pipeline d'Anonymisation Contextuelle (Volet Individuel)

> **Mission** : audit éthique complet du dataset Adult Income enrichi de
> commentaires manager. Datasheet duo signée. La phase d'anonymisation
> personnelle se fera en async (notebook séparé).

Auteur : `Joelle` Date : `17/06/2026`

## 0. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
DATA_DIR = Path("../data")
FULL_PATH = DATA_DIR / "adult_income_with_comments.csv"
SAMPLE_PATH = DATA_DIR / "audit_sample.csv"

sns.set_theme(style="whitegrid")

## 1. Exploration des PII dans manager_comments

In [2]:
import pandas as pd
import re

df_audit = pd.read_csv(SAMPLE_PATH)

# Expressions régulières avancées pour capturer l'exhaustivité des PII détectées
pii_patterns = {
    "Email": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
    "Téléphone": r'\b\d{3}[.-]\d{3}[.-]\d{4}\b',
    "Date (YYYY-MM-DD)": r'\b\d{4}-\d{2}-\d{2}\b',
    "IBAN Partiel / Compte": r'\*\*\*\*\d{4}',
    "Ticket RH / ID": r'\bHR-\d{5}\b',
    # Cette Regex gère : Prénoms/Noms simples, composés (Gérard-Frédéric), et les particules (de, du, le)
    "Nom Suspect (Composé/Particule)": r'\b[A-ZÉÈÀÂÇ][a-zçéèàâüû]+(?:-[A-ZÉÈÀÂÇ][a-zçéèàâüû]+)?(?:\s+(?:de|du|le)\s+|\s+)[A-ZÉÈÀÂÇ][a-zçéèàâüû]+(?:-[A-ZÉÈÀÂÇ][a-zçéèàâüû]+)?\b'
}

# Initialisation des compteurs et exemples
pii_counts = {key: 0 for key in pii_patterns.keys()}
pii_samples = {key: set() for key in pii_patterns.keys()}

# Analyse de la colonne 'manager_comments'
for comment in df_audit['manager_comments'].dropna():
    for pii_type, pattern in pii_patterns.items():
        matches = re.findall(pattern, str(comment))
        if matches:
            pii_counts[pii_type] += len(matches)
            # Sauvegarde de 3 exemples uniques max pour l'affichage
            for m in matches:
                if len(pii_samples[pii_type]) < 3:
                    pii_samples[pii_type].add(m)

# Création du rapport final
df_report = pd.DataFrame({
    "Type de PII": pii_counts.keys(),
    "Total Détecté": pii_counts.values(),
    "Exemples Extraits": [", ".join(list(pii_samples[k])) for k in pii_patterns.keys()]
}).sort_values(by="Total Détecté", ascending=False)

print("\n========================================================================")
print("               BILAN DU COMPTAGE DES PII (TÂCHE 6)                      ")
print("========================================================================")
print(df_report.to_string(index=False))
print("========================================================================")


               BILAN DU COMPTAGE DES PII (TÂCHE 6)                      
                    Type de PII  Total Détecté                                                       Exemples Extraits
Nom Suspect (Composé/Particule)            399                               Daniel Murphy, Gregory Meyer, Dennis Huff
              Date (YYYY-MM-DD)             86                                      2025-08-03, 2025-07-13, 2025-02-11
                          Email             85 ashleyraymond@example.org, garciaglen@example.org, rcordova@example.net
                      Téléphone             60                                310.727.4959, 144.909.0438, 669.289.2298
                 Ticket RH / ID             60                                            HR-24680, HR-20125, HR-65938
          IBAN Partiel / Compte             39                                            ****1400, ****9722, ****8655


## 2. Mise en place spaCy NER (regex en complément)

In [104]:
import sys
import os
import pandas as pd

# Configuration du chemin vers src/
dossier_src = os.path.abspath(os.path.join("..", "src"))
if dossier_src not in sys.path:
    sys.path.append(dossier_src)

# Purge du cache pour charger la dernière version de ton fichier anonymise.py
if "anonymize" in sys.modules:
    del sys.modules["anonymize"]

from anonymize import audit_with_en_model, anonymize_comments

if os.path.exists(SAMPLE_PATH):
    df = pd.read_csv(SAMPLE_PATH).head(15)  # Échantillon de ~15 lignes
    
    print("AUDIT QUALITATIF (spaCy EN et Regex)\n")
    for idx, row in df.iterrows():
        text = row['manager_comments']
        res = audit_with_en_model(text)   
       
        print(f" Commentaire #{idx + 1}")
        print(f" Brut : {text}")
        print(f" - Capturé par spaCy EN : PERSON={res.get('PERSON', [])} | GPE={res.get('GPE', [])} | ORG={res.get('ORG', [])}")
        print(f" - Capturé par Regex    : EMAILS={res.get('EMAILS', [])} | PHONE={res.get('PHONE', [])} | IBAN={res.get('IBAN', [])}")
        
else:
    print(f"⚠️ Fichier introuvable à l'emplacement : {SAMPLE_PATH}")

AUDIT QUALITATIF (spaCy EN et Regex)

 Commentaire #1
 Brut : Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
 - Capturé par spaCy EN : PERSON=['Daniel Murphy', 'Gregory Meyer'] | GPE=[] | ORG=[]
 - Capturé par Regex    : EMAILS=['ashleyraymond@example.org'] | PHONE=[] | IBAN=[]
 Commentaire #2
 Brut : Dennis Huff requested a transfer. Approved by Courtney Miller. Reach them at 310.727.4959 for handover. Ticket HR-24680.
 - Capturé par spaCy EN : PERSON=['Dennis Huff', 'Courtney Miller'] | GPE=[] | ORG=[]
 - Capturé par Regex    : EMAILS=[] | PHONE=['310.727.4959'] | IBAN=[]
 Commentaire #3
 Brut : Strong year for Cynthia Hardy. Bonus wired to account ****1400. Validated by Brittany Garcia on 2025-02-11.
 - Capturé par spaCy EN : PERSON=['Cynthia Hardy', 'Brittany Garcia'] | GPE=[] | ORG=[]
 - Capturé par Regex    : EMAILS=[] | PHONE=[] | IBAN=['****1400']
 Commentaire #4
 Brut : Plan d'accompagnement ouvert pour Pierre

In [105]:
    print("\n Vérification avec anonymize_comments) ")
   
    for idx, row in df.iterrows():
        
        print(f"Commentaire #{idx + 1} ")
        print(f" 🔴 AVANT : {row['manager_comments']}")
        print(f" 🟢 APRÈS : {anonymize_comments(row['manager_comments'])}")


 Vérification avec anonymize_comments) 
Commentaire #1 
 🔴 AVANT : Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
 🟢 APRÈS : [INDIVIDU_1] flagged a conflict with [INDIVIDU_2]. Mediation scheduled. HR follow-up: [EMAIL].
Commentaire #2 
 🔴 AVANT : Dennis Huff requested a transfer. Approved by Courtney Miller. Reach them at 310.727.4959 for handover. Ticket HR-24680.
 🟢 APRÈS : [INDIVIDU_2] requested a transfer. Approved by [INDIVIDU_1]. Reach them at [PHONE] for handover. Ticket HR-24680.
Commentaire #3 
 🔴 AVANT : Strong year for Cynthia Hardy. Bonus wired to account ****1400. Validated by Brittany Garcia on 2025-02-11.
 🟢 APRÈS : Strong year for [INDIVIDU_2]. Bonus wired to account [IBAN]. Validated by [INDIVIDU_1] on 2025-02-11.
Commentaire #4 
 🔴 AVANT : Plan d'accompagnement ouvert pour Pierre Laporte. Coaching confié à Pauline Monnier (rcordova@example.net).
 🟢 APRÈS : Plan d'accompagnement ouvert pour [INDIVIDU_

**limite linguistique**
**Trou 1 : Le modèle anglais dévore le vocabulaire français (Commentaires #9, #10, #12)**
Commentaire #9 : * Brut : Alerte comportementale signalée par Valentine Ribeiro...
Anonymisé : Alerte [INDIVIDU_2] par [INDIVIDU_1]...
Ce qui s'est passé : Le modèle anglais a cru que le mot "comportementale" et le mot "signalée" étaient des noms propres (PERSON). Il a donc remplacé des verbes et des adjectifs français par des tags [INDIVIDU].
Commentaire #10 : * Brut : Prime versée à Paulette...
Anonymisé : Prime [INDIVIDU_3] à [INDIVIDU_1]...
Ce qui s'est passé : Même chose, "versée" a été détecté comme un être humain.

**Trou 2 : Les noms de famille composés français sont tronqués (Commentaires #9, #12, #13)**
Commentaire #9 : Noël Joubert de Lagarde est devenu un patchwork d'individus parce que spaCy n'a pas compris la particule "de Lagarde".
Commentaire #12 : Sabine Rey du Grondin a été complètement oubliée ou coupée en morceaux.
Commentaire #13 : Philippine de la Duval n'a pas du tout été anonymisée. Elle est restée intacte.

**Trou 3 : Faux positifs d'organisations et d'e-mails (Commentaires #13, #14)**
Commentaire #13 : Le sigle RAS (Rien À Signaler) a été transformé en [ORGANISATION].
Commentaire #14 : L'adresse e-mail william00@example.org a été transformée en [ORGANISATION] par spaCy avant que votre Regex d'e-mail n'ait pu s'exécuter !

**ANONYMISATION DE LA SÉLECTION AVEC FR_CORE_NEWS_MD sans changer anonymize.py** 

In [109]:
import os
import re
import pandas as pd
import spacy

# Chargement du modèle français de confiance
nlp_fr = spacy.load("fr_core_news_md")

# Regex de nettoyage
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
PHONE_RE = re.compile(r"\b\d{3}[.-]?\d{3}[.-]?\d{4}\b")
IBAN_PARTIAL_RE = re.compile(r"\*{2,}\d{4}")

SAMPLE_PATH = "../data/audit_sample.csv"

if os.path.exists(SAMPLE_PATH):
    df = pd.read_csv(SAMPLE_PATH)
    
    # Indices Python correspondants aux commentaires 9, 10, 12, 13, 14 (Index = Numéro - 1)
    indices_cibles = [8, 9, 11, 12]
    
    print("--- FOCUS : ANONYMISATION DE LA SÉLECTION AVEC FR_CORE_NEWS_MD ---\n")
    
    for idx in indices_cibles:
        if idx >= len(df):
            continue
            
        text = df.iloc[idx]['manager_comments']
        
        # 1. Pré-nettoyage des données structurées
        anonymized = EMAIL_RE.sub("[EMAIL]", text)
        anonymized = PHONE_RE.sub("[PHONE]", anonymized)
        anonymized = IBAN_PARTIAL_RE.sub("[IBAN]", anonymized)
        
        # 2. Extraction des entités avec le modèle de langue français
        doc = nlp_fr(anonymized)
        
        # Extraction de toutes les entités identifiées (PERSON, LOC, ORG)
        # en évitant de capturer par erreur nos tags regex existants
        entites = []
        for ent in doc.ents:
            val = ent.text.strip()
            if val and len(val) > 1 and not any(tag in val for tag in ["[EMAIL]", "[PHONE]", "[IBAN]"]):
                if val not in entites:
                    entites.append(val)
        
        # Tri décroissant pour ne pas casser les noms composés (ex: Noël Joubert de Lagarde)
        entites.sort(key=len, reverse=True)
        
        # 3. Remplacement séquentiel par des tags d'identification
        for i, entite in enumerate(entites, start=1):
            anonymized = re.sub(r'\b' + re.escape(entite) + r'\b', f"[INDIVIDU_{i}]", anonymized)
            
        print(f"Commentaire #{idx + 1} 🇫🇷 [Pipeline FR]")
        print(f" AVANT : {text}")
        print(f" APRÈS : {anonymized}")
        print("-" * 70)
else:
    print(f"⚠️ Fichier introuvable : {SAMPLE_PATH}")

--- FOCUS : ANONYMISATION DE LA SÉLECTION AVEC FR_CORE_NEWS_MD ---

Commentaire #9 🇫🇷 [Pipeline FR]
 AVANT : Alerte comportementale signalée par Valentine Ribeiro au sujet de Noël Joubert de Lagarde. Suivi RH : harpermonica@example.net.
 PRÈS : Alerte comportementale signalée par [INDIVIDU_2] au sujet de [INDIVIDU_1]. Suivi RH : [EMAIL].
----------------------------------------------------------------------
Commentaire #10 🇫🇷 [Pipeline FR]
 AVANT : Prime versée à Paulette Gosselin-Delahaye sur le compte ****9722. Contrôle : Odette Thomas.
 PRÈS : [INDIVIDU_3] versée à [INDIVIDU_1] sur le compte [IBAN]. Contrôle : [INDIVIDU_2].
----------------------------------------------------------------------
Commentaire #12 🇫🇷 [Pipeline FR]
 AVANT : Alerte comportementale signalée par Jacqueline Noël-Buisson au sujet de Sabine Rey du Grondin. Suivi RH : cody76@example.com.
 PRÈS : Alerte comportementale signalée par [INDIVIDU_1] au sujet de [INDIVIDU_2]. Suivi RH : [EMAIL].
-----------------------

In [ ]:
Les commentaires #9 et #12 sont désormais parfaits : les noms à particule complexes ("Noël Joubert de Lagarde", "Sabine Rey du Grondin") et les noms composés sont nettoyés d'un seul bloc, et les faux positifs ("comportementale", "signalée") ont complètement disparu.

Cependant, en analysant vos résultats, on détecte les deux trous à régler :
Commentaire #10 : Le mot "Prime" a été mangé
Après : [INDIVIDU_3] versée à [INDIVIDU_1]...
Le modèle français a confondu le mot "Prime" (placé en début de phrase avec une majuscule) avec un prénom/nom de famille.

Commentaire #13 : Le mot "Manager" a été mangé
Après : RAS pour [INDIVIDU_2] cette année. [INDIVIDU_3] : [INDIVIDU_1].
"Manager" suivi de deux points a été interprété à tort comme une entité par spaCy.

## 3. Stratégie d'anonymisation

## 1. Choix et Défense de la Stratégie : La Généralisation Contextuelle

Pour traiter la colonne textuelle `manager_comments`, j'ai opté pour la **Généralisation Contextuelle** (remplacement des PII par des tags qualifiés typés comme `[EMAIL]`, `[PHONE]`, `[NOM]`, `[TICKET_RH]`) plutôt que pour les approches alternatives. 

### Pourquoi avoir éliminé les autres stratégies ?
*   **La Suppression brute (`[REDACTED]`) :** Détruit la sémantique de la phrase. Si un texte contient *"Le manager [REDACTED] a contacté l'employé [REDACTED] à l'adresse [REDACTED]"*, le texte perd toute utilité pour un futur modèle de NLP.
*   **La Substitution synthétique (Faker) :** Générer de faux noms est une excellente méthode pour de la démonstration visuelle, mais hautement dangereuse en Data Science. Un modèle pourrait associer des biais à ces noms générés aléatoirement, polluant ainsi l'apprentissage.
*   **Le Hachage (`SHA-256`) :** Le hachage de données à faible entropie (comme un numéro de téléphone ou un nom) est vulnérable aux attaques par dictionnaire. De plus, au sens de la CNIL et du RGPD, le hachage est une pseudonymisation, pas une anonymisation. Le risque de réidentification reste entier si la table de correspondance est compromis.

### Justification du compromis Utilité / Confidentialité :
La généralisation permet de **sanitiser mathématiquement** le fichier tout en conservant la structure grammaticale, la syntaxe et l'intention du message managérial. Les modèles de NLP peuvent continuer à travailler sur les structures de phrases sans jamais toucher à une donnée nominative.

## 2. Cartographie des données : Ce qui est masqué vs Ce qui reste lisible

### Ce qui est masqué (Risque Critique) :
*   **Les identifiants directs et indirects :** Noms, prénoms, adresses e-mails, numéros de téléphone.
*   **Les traces opérationnelles :** reliquats de coordonnées bancaires (`****1400`). Donnée sans aucune valeur pour l'évaluation éthique mais représente un risque cyber de pivotement informatique majeur.

### Ce qui reste lisible (Utilité Métier) :
*   **Les indicateurs de performance et de contexte :** Le vocabulaire lié aux évaluations, aux objectifs, aux retards, ou aux compétences.
*   **Vocabulaire Métier :** Le vocabulaire lié aux métiers (Manager, Prime). Mots-clés contextuels. Indispensables pour comprendre l'évaluation.
*   **Les structures intersectionnelles et tabulaires :** Les variables `sex`, `race`, et `income` de la ligne correspondante restent inchangées dans le volet tabulaire. Donc on peut auditer le biais salarial tout en protégeant la vie privée textuelle des individus.

## 4. Bascule vers Microsoft Presidio

## Ouverture Industrielle : La bascule vers Microsoft Presidio

Pour dépasser ces limites en production chez Athéna RH, l'alternative industrielle de référence est **Microsoft Presidio**. 

### Pourquoi Presidio est-il supérieur pour notre cas d'usage ?
1.  **Architecture Multilingue Native :** Presidio intègre un orchestrateur capable de lier plusieurs modèles NER en parallèle (un modèle spaCy FR + un modèle spaCy EN). Il résout ainsi instantanément notre problème des 12% de fuites francophones.
2.  **Pipeline d'Anonymisation Découplé :** Presidio sépare strictement l'étape de détection (*Analyzer*) de l'étape de modification (*Anonymizer*). Cela permet d'appliquer des stratégies différenciées par type d'entité (ex : *redact* pour les IBAN, *replace avec un tag* pour les noms, *hash* pour les IDs utilisateurs) via un simple fichier de configuration JSON.
3.  **Gestion des Triggers Contextuels :** Presidio utilise des listes de mots-clés contextuels (*Context Words*). Si le mot *"téléphone"* ou *"mobile"* est détecté à proximité d'une suite de chiffres, le score de confiance de l'entité augmente, limitant drastiquement les faux négatifs que nos Regex rigides n'auraient pas capturés.

In [142]:
import os
import re
import pandas as pd
import spacy
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# --- CONFIGURATION MULTILINGUE EXPLICITE ---
configuration_multilingue = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "en", "model_name": "en_core_web_md"},
        {"lang_code": "fr", "model_name": "fr_core_news_md"}
    ],
}

provider = NlpEngineProvider(nlp_configuration=configuration_multilingue)
nlp_engine = provider.create_engine()
# On déclare explicitement le support des deux langues
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["en", "fr"])
anonymizer = AnonymizerEngine()

def anonymize_manager_comments_presidio(text: str, lang: str = "fr") -> str:
    if not isinstance(text, str) or pd.isna(text) or not text.strip():
        return ""

    # Étape 1 : Analyse (Correction des noms d'entités officiels de Presidio)
    analysis_results = analyzer.analyze(
        text=text,
        language=lang,
        entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IBAN"]
    )
    
    # Étape 2 : Extraction ordonnée des noms propres (Chronologie de gauche à droite)
    detected_names = []
    for res in sorted(analysis_results, key=lambda x: x.start):
        if res.entity_type == "PERSON":
            name_extracted = text[res.start:res.end].strip()
            if name_extracted not in detected_names and len(name_extracted) > 1:
                detected_names.append(name_extracted)

    # CORRECTION : L'opérateur personnalisé Presidio reçoit l'objet original d'analyse
    def dynamic_person_operator(original_text_or_dict):
        # Presidio passe parfois une chaîne ou parfois l'objet analysé selon la version
        target_text = getattr(original_text_or_dict, 'text', str(original_text_or_dict)).strip()
        if target_text in detected_names:
            idx = detected_names.index(target_text) + 1
            return f"[INDIVIDU_{idx}]"
        return "[INDIVIDU]"

    # Étape 3 : Mapping de la stratégie (Correction des clés d'opérateurs)
    operators_mapping = {
        "PERSON": OperatorConfig("custom", {"lambda": dynamic_person_operator}),
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
        "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "[PHONE]"}),
        "IBAN": OperatorConfig("replace", {"new_value": "[IBAN]"}),
    }

    # Étape 4 : Remplacement chirurgical par coordonnées
    anonymized_output = anonymizer.anonymize(
        text=text,
        analyzer_results=analysis_results,
        operators=operators_mapping
    )
    
    final_text = anonymized_output.text

    # Étape 5 : Nettoyage résiduel (Patterns masqués type ****0122)
    final_text = re.sub(r'\*+\d{4}', '[IBAN]', final_text)

    return final_text

In [143]:
if os.path.exists(SAMPLE_PATH):
    df_presidio = pd.read_csv(SAMPLE_PATH).head(15)
    
    print("--- CONFRONTATION DES STRATÉGIES ---\n")
    for idx, row in df_presidio.iterrows():
        text = row['manager_comments']
        
        # Routage linguistique rapide pour alimenter Presidio
        is_fr = any(w in text.lower() for w in ["alerte", "comportementale", "signalée", "sujet", "prime"])
        lang_code = "fr" if is_fr else "en"
        
        presidio_res = anonymize_manager_comments_presidio(text, lang=lang_code)
        
        print(f"Commentaire #{idx + 1}")
        print(f" - Brut : {text}")
        print(f" - Spacy + Regex : {anonymize_comments(text)}") # Votre ancienne fonction
        print(f" - PRESIDIO      : {presidio_res}")
        print("-" * 80)

--- CONFRONTATION DES STRATÉGIES ---

Commentaire #1
 - Brut : Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
 - Spacy + Regex : [INDIVIDU_1] flagged a conflict with [INDIVIDU_2]. Mediation scheduled. HR follow-up: [EMAIL].
 - PRESIDIO      : [INDIVIDU_1] flagged a conflict with [INDIVIDU_2]. Mediation scheduled. HR follow-up: [EMAIL].
--------------------------------------------------------------------------------
Commentaire #2
 - Brut : Dennis Huff requested a transfer. Approved by Courtney Miller. Reach them at 310.727.4959 for handover. Ticket HR-24680.
 - Spacy + Regex : [INDIVIDU_2] requested a transfer. Approved by [INDIVIDU_1]. Reach them at [PHONE] for handover. Ticket HR-24680.
 - PRESIDIO      : [INDIVIDU_1] requested a transfer. Approved by [INDIVIDU_2]. Reach them at [PHONE] for handover. Ticket HR-24680.
--------------------------------------------------------------------------------
Commentaire #3
 - Br

**La fin du "Bug d'inversion des index" (Commentaires #2, #3, #4, #14)**
Le problème (spaCy) : l'attribution des index était désordonnée. Dans le commentaire #2, Courtney Miller (nom plus long) devenait [INDIVIDU_1] bien qu'elle arrive en second dans la phrase, devant Dennis Huff ([INDIVIDU_2]).
La correction Presidio : Grâce au tri par coordonnées spatiales (x.start), l'indexation respecte l'ordre de lecture naturel. Dennis Huff est bien [INDIVIDU_1] et Courtney Miller est [INDIVIDU_2]. La cohérence narrative est préservée.

**Sauvetage de la structure grammaticale française (Commentaires #9, #12)**
Le problème (spaCy) : Le modèle anglais coupait le texte en confondant des verbes ou des adjectifs avec des noms propres ("Alerte [INDIVIDU_2] par...").
La correction Presidio : Le texte métier conserve tout son sens ("Alerte comportementale signalée par..."). Seuls les vrais collaborateurs sont ciblés.

**Nettoyage des cas complexes et des particules (Commentaire #13)**
Le problème historique (spaCy) : spaCy + Regex passait à côté de Philippine de la Duval car les particules complexes perturbent l'extraction de tokens classique, et il hallucinait un tag [ORGANISATION] sur le mot "RAS".
La correction Presidio : Le mot "RAS" est correctement ignoré, et le nom de famille à particule est intégralement masqué.

In [152]:
import os
import pandas as pd

# 1. Définition des chemins de fichiers
INPUT_PATH = "../data/audit_sample.csv"
# Remplacez 'valentine' par votre propre prénom en minuscules
OUTPUT_PATH = "../data/audit_sample_anonymized_joelle.csv"

# 2. Chargement du fichier source
if os.path.exists(INPUT_PATH):
# 1. Lecture du fichier initial
    df = pd.read_csv(INPUT_PATH)
    print(f"✅ Fichier source chargé ({len(df)} lignes).")
    
    # 2. Remplacement ÉCRASANT de la colonne d'origine
    print("⏳ Anonymisation et remplacement de la colonne 'manager_comments'...")
    df["manager_comments"] = df["manager_comments"].apply(anonymize_manager_comments_presidio)
    
    # 3. Sauvegarde dans le nouveau fichier demandé pour la Tâche 9
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    
    print(f"✨ Fichier sécurisé généré avec succès : {OUTPUT_PATH}")
    print("\n👀 Vérification de la colonne remplacée (Aperçu) :")
    print(df["manager_comments"].head(15))

else:
    print(f"❌ Erreur : Le fichier source est introuvable à l'emplacement {os.path.abspath(INPUT_PATH)}")
    print("Veuillez vérifier que le fichier d'origine s'appelle bien 'audit_sample.csv' dans votre dossier 'data'.")

✅ Fichier source chargé (200 lignes).
⏳ Anonymisation et remplacement de la colonne 'manager_comments'...
✨ Fichier sécurisé généré avec succès : ../data/audit_sample_anonymized_joelle.csv

👀 Vérification de la colonne remplacée (Aperçu) :
0     [INDIVIDU_1] a conflict with [INDIVIDU_2]. Med...
1     [INDIVIDU_1] requested a transfer. Approved by...
2     Strong year for [INDIVIDU_1]. Bonus wired to a...
3     Plan d'accompagnement ouvert pour [INDIVIDU_1]...
4     Strong year for [INDIVIDU_1]. Bonus wired to a...
5     Annual review for [INDIVIDU_1]: solid contribu...
6     Behavioral concern raised by colleague [INDIVI...
7     [INDIVIDU_1] requested a transfer. Approved by...
8     Alerte comportementale signalée par [INDIVIDU_...
9     Prime versée à [INDIVIDU_1] sur le compte [IBA...
10    Performance below target for [INDIVIDU_1]. Act...
11    Alerte comportementale signalée par [INDIVIDU_...
12    RAS pour [INDIVIDU_1] cette année. Manager : [...
13    Onboarding complete for Je

In [153]:
# Installation du moteur si nécessaire
# !pip install pyarrow

OUTPUT_PARQUET = "../data/audit_sample_anonymized_joelle.parquet"

# Exportation ultra-compressée avec le moteur PyArrow
df.to_parquet(
    OUTPUT_PARQUET,
    engine="pyarrow",
    compression="snappy",  # Équilibre parfait entre vitesse et compression
    index=False            # Évite d'ajouter une colonne d'index inutile
)

print(f"✅ Fichier Parquet industriel généré avec succès à l'emplacement : {OUTPUT_PARQUET}")

✅ Fichier Parquet industriel généré avec succès à l'emplacement : ../data/audit_sample_anonymized_joelle.parquet
